# 2.2. Источники данных и характеристики

**Тема:** Исследование поведения покупателей в онлайн-магазинах для обнаружения паттернов спроса и предпочтений.

Цель работы - подобрать источники данных разных типов и определить характеристики, которые помогут изучить спрос и предпочтения покупателей.

In [26]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from google.colab import files
from IPython.display import display

In [31]:
uploaded = files.upload()

print("Загруженные файлы:")
for name in uploaded:
    print(name)

data_file = next(
    (name for name in uploaded if name.lower() == "data.csv"),
    None
)

reviews_file = next(
    (name for name in uploaded
     if "womens clothing e-commerce reviews" in name.lower()),
    None
)

if data_file is None:
    raise FileNotFoundError("Не найден файл data.csv")

if reviews_file is None:
    raise FileNotFoundError(
        "Не найден файл Womens Clothing E-Commerce Reviews.csv"
    )

df = pd.read_csv(data_file, encoding="ISO-8859-1")
reviews = pd.read_csv(reviews_file)

print("\nОба датасета загружены.")
print("Покупки:", df.shape)
print("Отзывы:", reviews.shape)

Saving Womens Clothing E-Commerce Reviews.csv to Womens Clothing E-Commerce Reviews.csv
Saving data.csv to data.csv
Загруженные файлы:
Womens Clothing E-Commerce Reviews.csv
data.csv

Оба датасета загружены.
Покупки: (541909, 8)
Отзывы: (23486, 11)


## 1. Реляционные данные - E-Commerce Data

Источник: [Kaggle](https://www.kaggle.com/datasets/carrie1/ecommerce-data)

Датасет содержит информацию о покупках в интернет-магазине.

Категориальные признаки: товар, покупатель, страна.

Числовые признаки: количество товаров и цена.

Временной признак: дата покупки.

In [32]:
display(df.head())

print("Количество записей:", len(df))
print("Количество разных товаров:", df["Description"].nunique())
print("Количество стран:", df["Country"].nunique())
print("Средняя цена товара:", round(df["UnitPrice"].mean(), 2))

print("\nПять наиболее часто встречающихся товаров:")
display(df["Description"].value_counts().head(5))

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


Количество записей: 541909
Количество разных товаров: 4223
Количество стран: 38
Средняя цена товара: 4.61

Пять наиболее часто встречающихся товаров:


,count
Description,
WHITE HANGING HEART T-LIGHT HOLDER,2369
REGENCY CAKESTAND 3 TIER,2200
JUMBO BAG RED RETROSPOT,2159
PARTY BUNTING,1727
LUNCH BAG RED RETROSPOT,1638


### Источник 2. Иерархические данные XML

Для изучения иерархических данных используется XML-экспорт демонстрационного магазина WooCommerce. Источник содержит каталог товаров и сведения, характерные для интернет-магазина.

Основные характеристики:
- название товара;
- категория товара;
- артикул (SKU);
- цена;
- описание;
- сведения о наличии товара;
- информация о характеристиках и вариантах товаров.


In [52]:
from google.colab import files

uploaded_xml = files.upload()

Saving sample_products.xml to sample_products.xml


In [56]:
import xml.etree.ElementTree as ET
import pandas as pd

xml_file = "sample_products.xml"

tree = ET.parse(xml_file)
root = tree.getroot()

ns = {
    "wp": "http://wordpress.org/export/1.2/"
}

products = []

for item in root.findall(".//item"):

    post_type = item.findtext(
        "wp:post_type",
        default="",
        namespaces=ns
    )

    if post_type != "product":
        continue

    title = item.findtext("title", default="")

    categories = [
        category.text
        for category in item.findall("category")
        if category.attrib.get("domain") == "product_cat"
        and category.text
    ]

    meta = {}

    for postmeta in item.findall("wp:postmeta", ns):
        key = postmeta.findtext(
            "wp:meta_key",
            namespaces=ns
        )
        value = postmeta.findtext(
            "wp:meta_value",
            namespaces=ns
        )

        if key and value and key not in meta:
            meta[key] = value

    products.append({
        "Название": title,
        "Категория": ", ".join(categories),
        "Артикул": meta.get("_sku"),
        "Цена": meta.get("_price") or meta.get("_regular_price"),
        "Наличие": meta.get("_stock_status"),
        "Продажи": meta.get("total_sales")
    })

products_df = pd.DataFrame(products)

print("Количество товаров:", len(products_df))

display(products_df.head(10))

Количество товаров: 18


,Название,Категория,Артикул,Цена,Наличие,Продажи
0,V-Neck T-Shirt,Tshirts,woo-vneck-tee,15,instock,0
1,Hoodie,Hoodies,woo-hoodie,42,instock,0
2,Hoodie with Logo,Hoodies,woo-hoodie-with-logo,45,instock,0
3,T-Shirt,Tshirts,woo-tshirt,18,instock,0
4,Beanie,Accessories,woo-beanie,18,instock,0
5,Belt,Accessories,woo-belt,55,instock,0
6,Cap,Accessories,woo-cap,16,instock,0
7,Sunglasses,Accessories,woo-sunglasses,90,instock,0
8,Hoodie with Pocket,Hoodies,woo-hoodie-with-pocket,35,instock,0
9,Hoodie with Zipper,Hoodies,woo-hoodie-with-zipper,45,instock,0


## 3. Данные API в формате JSON

Источник: [DummyJSON Products API](https://dummyjson.com/docs/products)

API позволяет получать информацию о товарах в формате JSON.

Категориальные признаки: название и категория товара.

Числовые признаки: цена и рейтинг.

In [34]:
json_url = "https://dummyjson.com/products?limit=5"

response = requests.get(json_url, timeout=20)
response.raise_for_status()

data = response.json()
products = pd.DataFrame(data["products"])

display(products[["title", "category", "price", "rating"]])

,title,category,price,rating
0,Essence Mascara Lash Princess,beauty,9.99,2.56
1,Eyeshadow Palette with Mirror,beauty,19.99,2.86
2,Powder Canister,beauty,14.99,4.64
3,Red Lipstick,beauty,12.99,4.36
4,Red Nail Polish,beauty,8.99,4.32


## 4. Неструктурированные данные - отзывы покупателей

Источник: [Kaggle](https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews)

Датасет содержит реальные отзывы покупателей об одежде.

Категориальные признаки: товар, отдел и класс одежды.

Числовые признаки: оценка товара, возраст покупателя и количество положительных откликов.

Текстовый признак: содержание отзыва.

Отзывы помогают понять отношение покупателей к товарам.

In [51]:
display(reviews.head())

print("Количество отзывов:", len(reviews))
print("Средняя оценка:", round(reviews["Rating"].mean(), 2))

print("\nКоличество отзывов по оценкам:")
display(reviews["Rating"].value_counts().sort_index())

print("\nПримеры реальных отзывов:")
display(reviews[["Review Text", "Rating"]].dropna().head(5))

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


Количество отзывов: 23486
Средняя оценка: 4.2

Количество отзывов по оценкам:


,count
Rating,
1,842
2,1565
3,2871
4,5077
5,13131



Примеры реальных отзывов:


,Review Text,Rating
0,Absolutely wonderful - silky and sexy and comf...,4
1,Love this dress! it's sooo pretty. i happene...,5
2,I had such high hopes for this dress and reall...,3
3,"I love, love, love this jumpsuit. it's fun, fl...",5
4,This shirt is very flattering to all due to th...,5


## Аналитический портрет объекта исследования

Для исследования поведения покупателей выбраны следующие характеристики:

- Категориальные: товар, категория, страна покупателя.
- Числовые: цена, количество покупок, рейтинг товара.
- Временные: дата покупки.
- Текстовые: отзывы покупателей.

Табличные данные позволяют изучать покупки и спрос. XML содержит сведения о продуктах, JSON API предоставляет характеристики товаров, а отзывы помогают изучать мнение покупателей.

Таким образом, выбранные источники содержат характеристики, которые можно использовать для исследования поведения покупателей и их предпочтений.